In [ ]:
"""
Process Data
======================================

Builds analysis‑ready grids and treatment sites. Part 1 joins MTBS and FIREDpy to the state grid with gpd.sjoin to compute burn‑year metrics, saving via GeoDataFrame.to_file. Part 2 computes trail_length per cell using gpd.overlay and geometry.length; Part 3 assigns PADUS management and counts proxies with gpd.overlay and gpd.sjoin. Part 4 prepares California Rx treatment sites: reproject with to_crs, intersect via gpd.sjoin, remove roads/urban using buffer + unary_union + geometry.difference, clip to PADUS, and write outputs.

"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from pathlib import Path

# Resolve directory of this file (works in .py; falls back to CWD in notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

# File paths (relative to this notebook)
colorado_grid_path = (HERE / "../data/processed_colorado_grid_final.shp").resolve()
mtbs_path = (HERE / "../data/mtbs_perims_2020_2024.shp").resolve()
firedpy_path = (HERE / "../data/fired_conus_ak_2000_to_2024_events.shp").resolve()
output_path = (HERE / "../outputs/colorado_grid_with_burn_history.shp").resolve()

# Load data
print("Loading Colorado grid and fire perimeters...")
grid = gpd.read_file(colorado_grid_path)
mtbs = gpd.read_file(mtbs_path)
fired = gpd.read_file(firedpy_path)

# Ensure CRS match
crs = grid.crs
mtbs = mtbs.to_crs(crs)
fired = fired.to_crs(crs)

# Extract year from MTBS
print("Extracting MTBS ignition years...")
mtbs["ig_year"] = pd.to_datetime(mtbs["Ig_Date"], errors="coerce").dt.year
mtbs = mtbs[(mtbs["ig_year"] >= 2000) & (mtbs["ig_year"] <= 2024)]

# Ensure FIREDpy has a valid 'ig_year' column already
fired = fired[(fired["ig_year"] >= 2000) & (fired["ig_year"] <= 2024)]

# --- Spatial Join: MTBS
print("Joining MTBS perimeters to grid...")
mtbs_join = gpd.sjoin(grid, mtbs[["ig_year", "geometry"]], how="left", predicate="intersects")
mtbs_grouped = mtbs_join.groupby(mtbs_join.index)["ig_year"].apply(set).to_dict()

# --- Spatial Join: FIREDpy
print("Joining FIREDpy perimeters to grid...")
fired_join = gpd.sjoin(grid, fired[["ig_year", "geometry"]], how="left", predicate="intersects")
fired_grouped = fired_join.groupby(fired_join.index)["ig_year"].apply(set).to_dict()

# --- Combine MTBS + FIREDpy, compute burn metrics
print("Combining burn history and computing metrics...")
burn_years_dict = {}

for idx in range(len(grid)):
    mtbs_years = mtbs_grouped.get(idx, set())
    fired_years = fired_grouped.get(idx, set()) - mtbs_years  # remove duplicates
    all_years = sorted(y for y in mtbs_years.union(fired_years) if pd.notnull(y) and 2000 <= y <= 2024)

    if not all_years:
        burn_years_dict[idx] = {
            "last_year_burned": None,
            "prior_burn_years": "",
            "avg_burn_interval_years": None
        }
    else:
        last_year = all_years[-1]
        prior_years = [str(y) for y in all_years[:-1]]
        interval = round(25 / len(all_years), 2)  # Using full time span
    
        burn_years_dict[idx] = {
            "last_year_burned": last_year,
            "prior_burn_years": ",".join(prior_years),
            "avg_burn_interval_years": interval
        }

# --- Merge into grid
print("Merging results into grid...")
burn_df = pd.DataFrame.from_dict(burn_years_dict, orient="index")
grid = grid.reset_index(drop=True).join(burn_df)

# --- Save output
print(f"Saving updated grid with burn history to: {output_path}")
grid.to_file(output_path)
print("Done: Burn metrics added to grid.")


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Resolve directory of this file (works in .py; falls back to CWD in notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

# File paths (relative to this notebook)
colorado_grid_path = (HERE / "../outputs/colorado_grid_with_burn_history.shp").resolve()
trails_path = (HERE / "../data/Trails_USWest.shp").resolve()
output_grid_path = (HERE / "../outputs/colorado_grid_with_trail_length.shp").resolve()

# Load data
print("Loading pre-processed Colorado grid...")
grid = gpd.read_file(colorado_grid_path)
trails = gpd.read_file(trails_path)

# Reproject to projected CRS for accurate length measurement
if grid.crs.is_geographic:
    print("Reprojecting grid to UTM Zone 13N...")
    grid = grid.to_crs("EPSG:26913")

if trails.crs != grid.crs:
    print("Reprojecting trails to match grid CRS...")
    trails = trails.to_crs(grid.crs)

# Create a unique ID for each grid cell
grid["grid_id"] = grid.index

# Spatial intersection of trails with grid
print("Computing trail length within each grid cell...")
trail_intersections = gpd.overlay(trails, grid, how="intersection")

# Compute trail length for each intersected segment
trail_intersections["trail_length"] = trail_intersections.geometry.length

# Sum trail length for each grid_id
trail_lengths = trail_intersections.groupby("grid_id")["trail_length"].sum().reset_index()

# Merge trail lengths back into grid
grid = grid.merge(trail_lengths, on="grid_id", how="left")
grid["trail_length"] = grid["trail_length"].fillna(0)

# Save output
grid.to_file(output_grid_path)
print(f"Saved updated grid with trail length to: {output_grid_path}")


In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from pathlib import Path

# Resolve directory of this file (works in .py; falls back to CWD in notebooks)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

# **File Paths** (relative to this notebook)
processed_colorado_grid_path = (HERE / "../outputs/colorado_grid_with_trail_length.shp").resolve()
padus_path = (HERE / "../data/PADUS4_0Comb_CO.shp").resolve()
proxies_merged_path = (HERE / "../data/ProxiesMerged.shp").resolve()
output_grid_path = (HERE / "../outputs/colorado_grid_manag_prox_trail_access.shp").resolve()

# **Step 1: Load Data**
print("Loading Colorado grid, PADUS management, and proxiesmerged points...")
colorado_grid = gpd.read_file(processed_colorado_grid_path)
padus = gpd.read_file(padus_path)
proxies = gpd.read_file(proxies_merged_path)

# **Ensure CRS Consistency**
print("Ensuring CRS alignment...")
padus = padus.to_crs(colorado_grid.crs)
proxies = proxies.to_crs(colorado_grid.crs)

# **Step 2: Assign Management Type**
print("Assigning majority management type to each grid cell...")
grid_padus_intersect = gpd.overlay(colorado_grid, padus, how="intersection")

# Count most common management type per grid cell
management_counts = grid_padus_intersect.groupby("grid_id")["Mang_Name"].agg(lambda x: x.value_counts().idxmax()).reset_index()

# Merge back to Colorado grid
colorado_grid = colorado_grid.merge(management_counts, on="grid_id", how="left")

# **Step 3: Count Number of Points in Each Grid Cell**
print("Counting `proxiesmerged` points in each grid cell...")
proxies_within_grid = gpd.sjoin(proxies, colorado_grid, how="inner", predicate="within")

# Count points per grid cell
point_counts = proxies_within_grid.groupby("grid_id").size().reset_index(name="proxy_count")

# Merge point counts back into Colorado grid
colorado_grid = colorado_grid.merge(point_counts, on="grid_id", how="left")

# Fill NaN values (cells with no proxies) with 0
colorado_grid["proxy_count"].fillna(0, inplace=True)

# **Step 4: Save Updated Grid**
print(f"Saving updated Colorado grid to: {output_grid_path}")
colorado_grid.to_file(output_grid_path)

print("Colorado grid updated with PADUS management and proxiesmerged point counts!")


In [ ]:
#!/usr/bin/env python3
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
# File Paths (relative to this notebook)
# ──────────────────────────────────────────────────────────────────────────────
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

processed_california_grid_path = (HERE / "../outputs/california_grid_manag_prox_trail_access.shp").resolve()
burn_perimeter_path            = (HERE / "../data/cleaned_dissolved_clustered_rx_CA.shp").resolve()
trails_path                    = (HERE / "../data/Trails_USWest.shp").resolve()
padus_path                     = (HERE / "../data/PADUS_Comb_CA.shp").resolve()
urban_path                     = (HERE / "../data/tl_2020_us_uac20.shp").resolve()
roads_path                     = (HERE / "../data/tl_2023_08_prisecroads.shp").resolve()
output_treatment_path          = (HERE / "../outputs/processed_Rx_sites_CA_final.shp").resolve()

# ──────────────────────────────────────────────────────────────────────────────
# 1. Load everything
# ──────────────────────────────────────────────────────────────────────────────
print("loading files")
grid   = gpd.read_file(processed_california_grid_path)
fires  = gpd.read_file(burn_perimeter_path)
trails = gpd.read_file(trails_path)
padus  = gpd.read_file(padus_path)
urban  = gpd.read_file(urban_path)
roads  = gpd.read_file(roads_path)

# ──────────────────────────────────────────────────────────────────────────────
# 2. Reproject to the grid's CRS
# ──────────────────────────────────────────────────────────────────────────────
print("reprojecting grid")
crs = grid.crs
for df in [fires, trails, padus, urban, roads]:
    df.to_crs(crs, inplace=True)

# ──────────────────────────────────────────────────────────────────────────────
# 2.5 Check and prepare fire data columns
# ──────────────────────────────────────────────────────────────────────────────
print("checking and preparing fire data columns")
print(f"Available columns in fires: {fires.columns.tolist()}")

# Create the missing columns if they don't exist
if 'Incid_Name' not in fires.columns:
    fires['Incid_Name'] = 'Rx_' + fires.index.astype(str)
    print("Created Incid_Name column")

if 'Ig_Date' not in fires.columns:
    fires['Ig_Date'] = '2020-01-01'  # Default date for Rx burns
    print("Created Ig_Date column")

if 'BurnBndAc' not in fires.columns:
    # Calculate area in acres (assuming geometry is in meters)
    fires['BurnBndAc'] = fires.geometry.area / 4047
    print("Created BurnBndAc column from geometry")

# ──────────────────────────────────────────────────────────────────────────────
# 3. Identify which trails and grid‐cells intersect fires
# ──────────────────────────────────────────────────────────────────────────────
# 3.1 Fire metadata on trails
print("intersections")

trail_fire_join = (
    gpd.sjoin(trails, fires[["geometry","Incid_Name","Ig_Date","BurnBndAc"]],
              how="inner", predicate="intersects")
    .drop(columns="index_right")
)

# 3.2 Grid cells that intersect any fire
fire_cells = (
    gpd.sjoin(grid, fires[["geometry","Incid_Name","Ig_Date","BurnBndAc"]],
              how="inner", predicate="intersects")
)
fire_cells["intersect_trail"] = "No"

# 3.3 Grid cells that intersect those fire‐affected trails
trail_grid_join = (
    gpd.sjoin(grid, trail_fire_join[["geometry","Incid_Name","Ig_Date","BurnBndAc"]],
              how="inner", predicate="intersects")
)
trail_grid_join["intersect_trail"] = "Yes"

# Combine and resolve duplicates
combined = pd.concat([fire_cells, trail_grid_join], ignore_index=True)
combined = combined.sort_values(by=["grid_id","intersect_trail"])
combined.loc[combined.duplicated("grid_id", keep=False),"intersect_trail"] = "Both"
combined = combined.drop_duplicates(subset="grid_id", keep="last")

# ──────────────────────────────────────────────────────────────────────────────
# 4. Keep just the columns you need and flag as treated
# ──────────────────────────────────────────────────────────────────────────────
columns_to_keep = [
    "grid_id","Access_U_1","trail_leng", "Trail_Dens", "proxy_coun","Mang_Name",
    "Elevation","Slope","Temperatur","Precipitat",
    "Tree_Cover","Shrub_Cove","Grass_Cove",
    "last_year_","prior_burn","avg_burn_i",
    "geometry","intersect_trail","Incid_Name","Ig_Date","BurnBndAc"
]
treatment_sites = combined[columns_to_keep].copy()
treatment_sites["treatment"] = 1

# ──────────────────────────────────────────────────────────────────────────────
# 5. Subtract roads + urban, then drop any lines/collections
# ──────────────────────────────────────────────────────────────────────────────
print("buffering roads by 100m...")
roads_buffer = roads.buffer(100)

print("creating combined mask of roads + urban areas...")
clip_mask = unary_union(urban.geometry.tolist() + roads_buffer.geometry.tolist())

print("subtracting roads and urban areas from treatment geometries...")
treatment_sites["geometry"] = treatment_sites.geometry.difference(clip_mask)

# ──────────────────────────────────────────────────────────────────────────────
# 6. Clip to PADUS (polygon‐in‐polygon), drop empties
# ──────────────────────────────────────────────────────────────────────────────
print("clipping geometries to PADUS public lands...")
treatment_sites = gpd.overlay(treatment_sites, padus, how="intersection")

# Remove empty geometries and null geometries
treatment_sites = treatment_sites[~treatment_sites.is_empty & treatment_sites.geometry.notnull()]

# ──────────────────────────────────────────────────────────────────────────────
# 7. Save final shapefile
# ──────────────────────────────────────────────────────────────────────────────
print(f"Saving treatment sites to: {output_treatment_path}")
treatment_sites.to_file(output_treatment_path)
print("Treatment sites saved with clipped geometries.")
